# Eksperimen Perbandingan Rasio Split Training-Testing
## Pipeline PNYB — ARIMA dan LSTM per Produk

Notebook ini menjalankan eksperimen untuk membandingkan kinerja model pada
**7 rasio split** yang berbeda:

| Rasio | Training | Testing |
|-------|----------|---------|
| 80:20 | 80% bulan pertama | 20% bulan terakhir |
| 70:30 | 70% bulan pertama | 30% bulan terakhir |
| 60:40 | 60% bulan pertama | 40% bulan terakhir |
| 50:50 | 50% bulan pertama | 50% bulan terakhir |
| 40:60 | 40% bulan pertama | 60% bulan terakhir |
| 30:70 | 30% bulan pertama | 70% bulan terakhir |
| 20:80 | 20% bulan pertama | 80% bulan terakhir |

Setiap rasio menjalankan pipeline yang identik dengan `pnyb_pipeline.py`.
Pada akhir notebook, seluruh hasil dirangkum dalam satu tabel perbandingan.

Urutan eksekusi: jalankan setiap sel dari atas ke bawah secara berurutan.


## Sel 0 — Setup: Import Library dan Fungsi Pipeline

Sel ini memuat seluruh pustaka, mendefinisikan konstanta yang tidak berubah antar eksperimen, dan mendefinisikan fungsi `jalankan_eksperimen()` yang akan dipanggil berulang kali dengan nilai `SPLIT_PCT` yang berbeda-beda.

In [1]:
import pandas as pd
import numpy as np
import time
import tracemalloc
import warnings
import os

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

DATA_PATH        = "dataset_toko.csv"
RANDOM_SEED      = 42
SEQ_LEN          = 6
N_CLUSTER        = 5
N_WINDOW_ARIMA   = 24
CAP_FACTOR_ARIMA = 1.2
CAP_FACTOR_LSTM  = 1.5
MIN_BULAN        = 15
MIN_BULAN_ARIMA  = 20
MIN_BULAN_AKTIF  = 3
ADAPTIVE_WINDOW  = 3

ARIMA_ORDERS = [
    (1,1,1),(1,1,0),(0,1,1),(2,1,0),(0,1,2),(2,1,1),
    (1,1,2),(3,1,0),(0,1,3),(2,1,2),(1,2,1),(0,2,1),
]
ARIMA_SEASONAL_ORDERS = [(0,0,0,0),(1,1,0,12)]

KALENDER_LIBUR = {
    '2020-05':0.40,'2021-05':0.45,'2022-05':0.55,'2023-04':0.50,
    '2024-04':0.65,'2025-03':0.90,'2020-06':0.60,'2021-06':0.60,
    '2022-06':0.65,'2020-07':0.60,'2021-07':0.65,'2022-07':0.70,
    '2023-06':0.65,'2024-05':0.60,'2024-06':0.12,'2024-07':0.28,
    '2024-08':0.55,'2024-12':0.40,'2025-01':0.58,
}
BULAN_EKSKLUDE = ['2024-06','2024-07']
BULAN_ANOMALI  = BULAN_EKSKLUDE
FEATURES = ['lag1_r','lag2_r','lag3_r','lag6_r','lag12_r',
            'roll3_r','roll6_r','tren_3m','bulan','produk_id','faktor_libur']

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# =====================================================================
# FUNGSI HELPER (identik pnyb_pipeline.py)
# =====================================================================
def hitung_momentum(df_c):
    if len(df_c) < 3: return 1.0
    vals = df_c['qty'].values[-3:]
    if vals[0] > 0:
        return float(np.clip(1.0+((vals[-1]-vals[0])/vals[0])*0.10, 0.90, 1.10))
    return 1.0

def prediksi_arima_pnyb(bulan_pred, avail, produk_layak, best_orders_arima,
                         best_seasonal_arima, bias_correction, harga_rata2, max_qty_produk):
    fl = KALENDER_LIBUR.get(str(bulan_pred), 1.0)
    hasil = []
    for produk in produk_layak:
        df_p = avail[avail['Nama Produk']==produk].sort_values('bulan_period')
        df_c = df_p[~df_p['bulan_period'].astype(str).isin(BULAN_EKSKLUDE)]
        if len(df_c) > N_WINDOW_ARIMA: df_c = df_c.tail(N_WINDOW_ARIMA)
        fc = None
        if produk in best_orders_arima and len(df_c) >= 6:
            try:
                qty_v = df_c['qty'].clip(lower=0.1).values
                fl_v  = df_c['bulan_period'].astype(str).map(lambda x: KALENDER_LIBUR.get(x,1.0)).values
                ts_log= np.log1p(qty_v/np.maximum(fl_v,0.1))
                m = SARIMAX(ts_log, order=best_orders_arima[produk],
                            seasonal_order=best_seasonal_arima.get(produk,(0,0,0,0)),
                            enforce_stationarity=False,
                            enforce_invertibility=False).fit(disp=False)
                fc_log= float(np.clip(m.forecast(steps=1).iloc[0],-2.0,10.0))
                fc = max(0.0, float(np.expm1(fc_log))*fl)
                if bias_correction.get(produk,1.0)>0: fc=fc/bias_correction.get(produk,1.0)
                fc = fc*hitung_momentum(df_c)
                fc = min(max(0.0,fc), max_qty_produk.get(produk,fc)*CAP_FACTOR_ARIMA)
            except: fc=None
        if fc is None and len(df_c)>=1:
            recent=df_c['qty'].values[-min(6,len(df_c)):]
            bobot=np.array([1,2,3,4,5,6][-len(recent):],dtype=float)
            fc=float(np.average(recent,weights=bobot))*fl
        if fc is None: fc=float(df_c['qty'].median())*fl if len(df_c)>0 else 0.0
        hasil.append({'nama_produk':produk,
                      'pred_qty_arima':round(max(0.0,fc),2),
                      'pred_rev_arima':round(max(0.0,fc)*harga_rata2.get(produk,0),0)})
    if not hasil: return pd.DataFrame(columns=['nama_produk','pred_qty_arima','pred_rev_arima'])
    return pd.DataFrame(hasil)

def prediksi_lstm_pnyb(bulan_pred, avail, produk_layak, cluster_models,
                        cluster_scalers, cluster_bias, produk_cluster, le,
                        harga_rata2, median_qty_produk, max_qty_produk):
    fl=KALENDER_LIBUR.get(str(bulan_pred),1.0); bi=bulan_pred.month; hasil=[]
    for produk in produk_layak:
        if produk not in le.classes_: continue
        cid=produk_cluster.get(produk,0)
        if cid not in cluster_models or cid not in cluster_scalers: continue
        df_p=avail[avail['Nama Produk']==produk].sort_values('bulan_period')
        df_c=df_p[~df_p['bulan_period'].astype(str).isin(BULAN_EKSKLUDE)]
        if len(df_c)<SEQ_LEN+4: continue
        med_q=median_qty_produk.get(produk,1.0); qty_v=df_c['qty'].values; n=len(qty_v)
        seq_feats=[]
        for step in range(SEQ_LEN):
            idx=n-1-(SEQ_LEN-1-step)
            if idx<0: break
            def lag(k): return qty_v[max(idx-k,0)]/max(med_q,1.0)
            r3=float(np.mean(qty_v[max(0,idx-3):idx]))/max(med_q,1.0) if idx>=3 else lag(1)
            r6=float(np.mean(qty_v[max(0,idx-6):idx]))/max(med_q,1.0) if idx>=6 else r3
            tren=((qty_v[idx]-qty_v[max(0,idx-3)])/max(abs(qty_v[max(0,idx-3)]),1.0)) if idx>=3 else 0.0
            try:
                cb=df_c['bulan_period'].iloc[idx]; bs=cb.month; fls=KALENDER_LIBUR.get(str(cb),1.0)
            except: bs=bi; fls=fl
            seq_feats.append([lag(1),lag(2),lag(3),lag(6),lag(12),r3,r6,tren,bs,
                               float(le.transform([produk])[0]),fls])
        if len(seq_feats)<SEQ_LEN: continue
        seq_sc=cluster_scalers[cid].transform(
            np.array(seq_feats[-SEQ_LEN:])).reshape(1,SEQ_LEN,len(FEATURES))
        pred_r=float(np.expm1(float(np.clip(
            cluster_models[cid].predict(seq_sc,verbose=0)[0,0],0,None))))
        if cluster_bias.get(cid,1.0)>0: pred_r=pred_r/cluster_bias[cid]
        pred_qty=min(max(0.0,max(0.0,pred_r*med_q)*fl),
                     max_qty_produk.get(produk,pred_r*med_q)*CAP_FACTOR_LSTM)
        hasil.append({'nama_produk':produk,'pred_qty_lstm':round(pred_qty,2),
                      'pred_rev_lstm':round(pred_qty*harga_rata2.get(produk,0),0)})
    if not hasil: return pd.DataFrame(columns=['nama_produk','pred_qty_lstm','pred_rev_lstm'])
    return pd.DataFrame(hasil)

# =====================================================================
# FUNGSI UTAMA EKSPERIMEN
# =====================================================================
def jalankan_eksperimen(split_pct, df_master, label=None):
    """
    Menjalankan satu siklus penuh pipeline PNYB dengan rasio split tertentu.
    Mengembalikan dict berisi semua metrik hasil eksperimen.
    """
    if label is None:
        train_pct = int(split_pct * 100)
        test_pct  = 100 - train_pct
        label     = f"{train_pct}:{test_pct}"

    print(f"\n{'='*65}")
    print(f"  EKSPERIMEN SPLIT {label}")
    print(f"{'='*65}")

    tf.random.set_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    # ------------------------------------------------------------------
    # TAHAP 1-6: PREPROCESSING (sudah dijalankan di luar, pakai df_master)
    # ------------------------------------------------------------------
    df = df_master.copy()

    bulan_list  = sorted(df['bulan_period'].unique())
    split_idx   = int(len(bulan_list) * split_pct)
    bulan_train = bulan_list[:split_idx]
    bulan_test  = bulan_list[split_idx:]

    if len(bulan_train) < MIN_BULAN + 5:
        print(f"  [SKIP] Data training terlalu sedikit: {len(bulan_train)} bulan")
        return None
    if len(bulan_test) < 3:
        print(f"  [SKIP] Data testing terlalu sedikit: {len(bulan_test)} bulan")
        return None

    print(f"  Training : {len(bulan_train)} bulan ({bulan_train[0]} s.d. {bulan_train[-1]})")
    print(f"  Testing  : {len(bulan_test)} bulan ({bulan_test[0]} s.d. {bulan_test[-1]})")

    harga_rata2 = df.groupby('Nama Produk')['Harga Jual (IDR)'].mean().to_dict()

    # ------------------------------------------------------------------
    # TAHAP 7: AGREGASI
    # ------------------------------------------------------------------
    monthly_all = (
        df.groupby(['bulan_period','bulan','Nama Produk'])
        .agg(qty=('Jumlah Produk Dibeli','sum'), revenue=('item_revenue','sum'))
        .reset_index()
        .sort_values(['Nama Produk','bulan_period'])
    )
    monthly_train = monthly_all[monthly_all['bulan_period'].isin(bulan_train)].copy()
    monthly_test  = monthly_all[monthly_all['bulan_period'].isin(bulan_test)].copy()

    # ------------------------------------------------------------------
    # TAHAP 9: FILTER PRODUK LAYAK
    # ------------------------------------------------------------------
    produk_count      = monthly_train.groupby('Nama Produk')['bulan_period'].count()
    produk_layak_awal = produk_count[produk_count >= MIN_BULAN].index.tolist()
    bulan_train_bersih= [b for b in bulan_train if str(b) not in BULAN_ANOMALI]
    bulan_cek_aktif   = bulan_train_bersih[-MIN_BULAN_AKTIF:]

    def cek_aktif(produk):
        df_p = monthly_train[monthly_train['Nama Produk']==produk]
        return df_p[df_p['bulan_period'].isin(bulan_cek_aktif)]['qty'].sum() > 0

    produk_layak   = [p for p in produk_layak_awal if cek_aktif(p)]
    produk_arima_l = [p for p in produk_layak if produk_count[p] >= MIN_BULAN_ARIMA]

    if len(produk_layak) == 0:
        print(f"  [SKIP] Tidak ada produk layak setelah filter.")
        return None

    mean_qty_produk, median_qty_produk, max_qty_produk = {}, {}, {}
    for p in produk_layak:
        vals = monthly_train[
            (monthly_train['Nama Produk']==p) &
            (~monthly_train['bulan_period'].astype(str).isin(BULAN_ANOMALI))
        ]['qty'].values
        mean_qty_produk[p]   = max(float(vals.mean()), 1.0) if len(vals)>0 else 1.0
        median_qty_produk[p] = max(float(np.median(vals)), 1.0) if len(vals)>0 else 1.0
        max_qty_produk[p]    = max(float(vals.max()), 1.0) if len(vals)>0 else 1.0

    print(f"  Produk layak: {len(produk_layak)} (ARIMA: {len(produk_arima_l)})")

    # ------------------------------------------------------------------
    # TAHAP 13: TRAINING ARIMA
    # ------------------------------------------------------------------
    print(f"  Melatih ARIMA ({len(produk_arima_l)} produk)...")
    tracemalloc.start(); t0 = time.time()
    best_orders_arima, best_seasonal_arima, bias_correction = {}, {}, {}
    for produk in produk_arima_l:
        df_c = monthly_train[monthly_train['Nama Produk']==produk].sort_values('bulan_period')
        df_c = df_c[~df_c['bulan_period'].astype(str).isin(BULAN_EKSKLUDE)].tail(N_WINDOW_ARIMA)
        if len(df_c) < 10:
            bias_correction[produk] = 1.0; continue
        fl_v   = df_c['bulan_period'].astype(str).map(lambda x: KALENDER_LIBUR.get(x,1.0)).values
        ts_log = np.log1p(df_c['qty'].clip(lower=0.1).values/np.maximum(fl_v,0.1))
        best_aic, best_order, best_seasonal = np.inf, (1,1,1), (0,0,0,0)
        for base_ord in ARIMA_ORDERS:
            for seas_ord in ARIMA_SEASONAL_ORDERS:
                try:
                    m = SARIMAX(ts_log, order=base_ord, seasonal_order=seas_ord,
                                enforce_stationarity=False,
                                enforce_invertibility=False).fit(disp=False)
                    if m.aic < best_aic:
                        best_aic, best_order, best_seasonal = m.aic, base_ord, seas_ord
                except: pass
        best_orders_arima[produk]   = best_order
        best_seasonal_arima[produk] = best_seasonal
        bias_correction[produk]     = 1.0
    t_arima   = time.time() - t0
    mem_arima = tracemalloc.get_traced_memory()[1]/1024**2
    tracemalloc.stop()
    print(f"  ARIMA selesai: {t_arima:.1f} detik, {mem_arima:.1f} MB")

    # ------------------------------------------------------------------
    # TAHAP 11-12-14: KLASTERISASI + REKAYASA FITUR + TRAINING LSTM
    # ------------------------------------------------------------------
    sorted_prods   = sorted(produk_layak, key=lambda p: median_qty_produk[p])
    cluster_size   = max(1, len(sorted_prods)//N_CLUSTER)
    produk_cluster = {p: min(i//cluster_size, N_CLUSTER-1) for i,p in enumerate(sorted_prods)}

    le  = LabelEncoder()
    mtr = monthly_train[monthly_train['Nama Produk'].isin(produk_layak)].copy()
    mtr['produk_id']  = le.fit_transform(mtr['Nama Produk'])
    mtr['median_qty'] = mtr['Nama Produk'].map(median_qty_produk)
    mtr['cluster_id'] = mtr['Nama Produk'].map(produk_cluster)

    for lag in [1,2,3,6,12]:
        mtr[f'lag{lag}_r'] = (
            mtr.groupby('Nama Produk')['qty'].transform(lambda x: x.shift(lag))
            / mtr['median_qty'].clip(lower=1.0))
    for w, col in [(3,'roll3_r'),(6,'roll6_r')]:
        mtr[col] = (
            mtr.groupby('Nama Produk')['qty']
            .transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())
            / mtr['median_qty'].clip(lower=1.0))
    mtr['tren_3m'] = mtr.groupby('Nama Produk')['qty'].transform(
        lambda x: x.shift(1).rolling(3, min_periods=2).apply(
            lambda v: (v[-1]-v[0])/max(abs(v[0]),1), raw=True))
    mtr['faktor_libur'] = mtr['bulan_period'].astype(str).map(
        lambda x: KALENDER_LIBUR.get(x,1.0))
    mtr['log_rasio'] = np.log1p(
        (mtr['qty']/mtr['median_qty'].clip(lower=1.0)).clip(0,8.0))
    mtr_clean = mtr.dropna()
    mtr_clean = mtr_clean[~mtr_clean['bulan_period'].astype(str).isin(BULAN_EKSKLUDE)].copy()

    print(f"  Melatih LSTM ({N_CLUSTER} klaster, {len(mtr_clean):,} sampel bersih)...")
    tracemalloc.start(); t0 = time.time()
    cluster_models, cluster_scalers, cluster_bias = {}, {}, {}
    for cid in range(N_CLUSTER):
        df_cl = mtr_clean[mtr_clean['cluster_id']==cid].reset_index(drop=True)
        if len(df_cl) < 40: continue
        scaler = MinMaxScaler()
        X_sc   = scaler.fit_transform(df_cl[FEATURES].values.astype(float))
        y_all  = df_cl['log_rasio'].values
        X_seq, y_seq = [], []
        for _, grp in df_cl.groupby('Nama Produk'):
            local_idx = list(grp.sort_values('bulan_period').index)
            for i in range(len(local_idx)-SEQ_LEN):
                X_seq.append(X_sc[local_idx[i:i+SEQ_LEN]])
                y_seq.append(y_all[local_idx[i+SEQ_LEN]])
        if len(X_seq) < 20: continue
        X_3d, y_arr = np.array(X_seq), np.array(y_seq)
        model = Sequential([
            LSTM(64, input_shape=(SEQ_LEN,len(FEATURES)), return_sequences=True),
            BatchNormalization(), Dropout(0.15),
            LSTM(32, return_sequences=False),
            BatchNormalization(), Dropout(0.15),
            Dense(16, activation='relu'), Dense(1)
        ])
        model.compile(loss=tf.keras.losses.Huber(delta=1.0),
                      optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4))
        model.fit(X_3d, y_arr, epochs=500, batch_size=16, validation_split=0.2,
            callbacks=[
                tf.keras.callbacks.EarlyStopping(
                    monitor='val_loss', patience=40, restore_best_weights=True),
                tf.keras.callbacks.ReduceLROnPlateau(
                    monitor='val_loss', patience=15, factor=0.5, min_lr=1e-6, verbose=0),
            ], verbose=0)
        cluster_models[cid]  = model
        cluster_scalers[cid] = scaler
        cluster_bias[cid]    = 1.0
    t_lstm   = time.time()-t0
    mem_lstm = tracemalloc.get_traced_memory()[1]/1024**2
    tracemalloc.stop()
    print(f"  LSTM selesai : {t_lstm:.1f} detik, {mem_lstm:.1f} MB")

    # ------------------------------------------------------------------
    # TAHAP 15: EVALUASI ROLLING
    # ------------------------------------------------------------------
    print(f"  Evaluasi rolling {len(bulan_test)} bulan testing...")
    semua_hasil, log_bulan, mae_arima_rol, mae_lstm_rol = [], [], [], []
    for idx_b, bulan_pred in enumerate(bulan_test):
        avail = monthly_all[monthly_all['bulan_period'] < bulan_pred]
        df_a  = prediksi_arima_pnyb(bulan_pred, avail, produk_layak,
                                     best_orders_arima, best_seasonal_arima,
                                     bias_correction, harga_rata2, max_qty_produk)
        df_l  = prediksi_lstm_pnyb(bulan_pred, avail, produk_layak,
                                    cluster_models, cluster_scalers, cluster_bias,
                                    produk_cluster, le, harga_rata2,
                                    median_qty_produk, max_qty_produk)
        df_m = pd.merge(df_a, df_l, on='nama_produk', how='outer')
        df_m['pred_qty_arima'] = df_m['pred_qty_arima'].fillna(df_m['pred_qty_lstm'])
        df_m['pred_qty_lstm']  = df_m['pred_qty_lstm'].fillna(df_m['pred_qty_arima'])
        w_arima, w_lstm = 0.40, 0.60
        if idx_b >= ADAPTIVE_WINDOW and mae_arima_rol:
            ea=float(np.mean(mae_arima_rol[-ADAPTIVE_WINDOW:]))
            el=float(np.mean(mae_lstm_rol[-ADAPTIVE_WINDOW:]))
            if ea+el>0: w_arima=el/(ea+el); w_lstm=ea/(ea+el)
        df_m['pred_qty_ens'] = (w_arima*df_m['pred_qty_arima']+w_lstm*df_m['pred_qty_lstm']).clip(0)
        for suf in ['arima','lstm','ens']:
            df_m[f'pred_rev_{suf}']=df_m[f'pred_qty_{suf}']*df_m['nama_produk'].map(harga_rata2).fillna(0)
        aktual_b = monthly_test[monthly_test['bulan_period']==bulan_pred][
            ['Nama Produk','qty','revenue']
        ].rename(columns={'Nama Produk':'nama_produk','qty':'aktual_qty','revenue':'aktual_rev'})
        df_m = pd.merge(df_m, aktual_b, on='nama_produk', how='left').fillna(0)
        semua_hasil.append(df_m)
        df_tp = df_m[df_m['aktual_qty']>0]
        if len(df_tp)>0:
            mae_arima_rol.append(float(mean_absolute_error(df_tp['aktual_qty'],df_tp['pred_qty_arima'])))
            mae_lstm_rol.append(float(mean_absolute_error(df_tp['aktual_qty'],df_tp['pred_qty_lstm'])))
        log_bulan.append({
            'bulan'     : str(bulan_pred),
            'aktual'    : df_m['aktual_rev'].sum(),
            'pred_arima': df_m['pred_rev_arima'].sum(),
            'pred_lstm' : df_m['pred_rev_lstm'].sum(),
            'pred_ens'  : df_m['pred_rev_ens'].sum(),
        })

    df_log = pd.DataFrame(log_bulan)

    # ------------------------------------------------------------------
    # TAHAP 16: HITUNG METRIK
    # ------------------------------------------------------------------
    mae_a  = float(mean_absolute_error(df_log['aktual'], df_log['pred_arima']))
    rmse_a = float(np.sqrt(mean_squared_error(df_log['aktual'], df_log['pred_arima'])))
    mae_l  = float(mean_absolute_error(df_log['aktual'], df_log['pred_lstm']))
    rmse_l = float(np.sqrt(mean_squared_error(df_log['aktual'], df_log['pred_lstm'])))
    mae_e  = float(mean_absolute_error(df_log['aktual'], df_log['pred_ens']))
    rmse_e = float(np.sqrt(mean_squared_error(df_log['aktual'], df_log['pred_ens'])))

    # MAPE per bulan (hanya bulan dengan aktual > 0)
    df_nz = df_log[df_log['aktual'] > 0].copy()
    mape_a = float(((df_nz['pred_arima']-df_nz['aktual']).abs()/df_nz['aktual']).mean()*100) if len(df_nz)>0 else float('nan')
    mape_l = float(((df_nz['pred_lstm'] -df_nz['aktual']).abs()/df_nz['aktual']).mean()*100) if len(df_nz)>0 else float('nan')
    mape_e = float(((df_nz['pred_ens']  -df_nz['aktual']).abs()/df_nz['aktual']).mean()*100) if len(df_nz)>0 else float('nan')

    print(f"  ARIMA   : MAE={mae_a/1e6:.3f}jt  RMSE={rmse_a/1e6:.3f}jt  MAPE={mape_a:.1f}%")
    print(f"  LSTM    : MAE={mae_l/1e6:.3f}jt  RMSE={rmse_l/1e6:.3f}jt  MAPE={mape_l:.1f}%")
    print(f"  Ensemble: MAE={mae_e/1e6:.3f}jt  RMSE={rmse_e/1e6:.3f}jt  MAPE={mape_e:.1f}%")

    return {
        'label'          : label,
        'split_pct'      : split_pct,
        'n_bulan_train'  : len(bulan_train),
        'n_bulan_test'   : len(bulan_test),
        'periode_train'  : f"{bulan_train[0]} s.d. {bulan_train[-1]}",
        'periode_test'   : f"{bulan_test[0]} s.d. {bulan_test[-1]}",
        'n_produk_layak' : len(produk_layak),
        'n_produk_arima' : len(produk_arima_l),
        'n_sampel_lstm'  : len(mtr_clean),
        't_arima'        : round(t_arima, 1),
        'mem_arima'      : round(mem_arima, 1),
        't_lstm'         : round(t_lstm, 1),
        'mem_lstm'       : round(mem_lstm, 1),
        'mae_arima'      : mae_a,
        'rmse_arima'     : rmse_a,
        'mape_arima'     : mape_a,
        'mae_lstm'       : mae_l,
        'rmse_lstm'      : rmse_l,
        'mape_lstm'      : mape_l,
        'mae_ens'        : mae_e,
        'rmse_ens'       : rmse_e,
        'mape_ens'       : mape_e,
        'df_log'         : df_log,
        'waktu_arima_det': t_arima,
        'waktu_lstm_det' : t_lstm,
    }

print("Setup selesai. Fungsi jalankan_eksperimen() siap digunakan.")
print(f"Dataset: {DATA_PATH}")
print(f"Rasio yang akan diuji: 80:20, 70:30, 60:40, 50:50, 40:60, 30:70, 20:80")


Setup selesai. Fungsi jalankan_eksperimen() siap digunakan.
Dataset: dataset_toko.csv
Rasio yang akan diuji: 80:20, 70:30, 60:40, 50:50, 40:60, 30:70, 20:80


---
## Sel 1 — Load dan Preprocessing Data Master

Data di-load dan di-preprocess satu kali di sel ini, kemudian DataFrame `df_master` yang sudah bersih digunakan berulang kali oleh setiap eksperimen. Tahapan preprocessing identik dengan Fase 1 `pnyb_pipeline.py`: parsing tanggal, filter status, pembersihan numerik, dan perhitungan `item_revenue`.


In [2]:
# =====================================================================
# LOAD DAN PREPROCESS SATU KALI (dipakai semua eksperimen)
# =====================================================================
print("Memuat dan memproses dataset...")

df_master = pd.read_csv(DATA_PATH, on_bad_lines='skip')

# Parsing tanggal
df_master['Tanggal Pembayaran'] = pd.to_datetime(
    df_master['Tanggal Pembayaran'], format='mixed', errors='coerce')
df_master = df_master.dropna(subset=['Tanggal Pembayaran'])

# Filter status pesanan selesai
df_master = df_master[df_master['Status Terakhir'] == 'Pesanan Selesai'].copy()

# Bersihkan kolom numerik
for col in ['Harga Jual (IDR)', 'Jumlah Produk Dibeli']:
    df_master[col] = pd.to_numeric(df_master[col], errors='coerce').fillna(0)

# Hitung item_revenue
df_master['item_revenue'] = (df_master['Harga Jual (IDR)'] * df_master['Jumlah Produk Dibeli']).clip(lower=0)

# Fitur waktu
df_master['bulan_period'] = df_master['Tanggal Pembayaran'].dt.to_period('M')
df_master['bulan']        = df_master['Tanggal Pembayaran'].dt.month

bulan_semua = sorted(df_master['bulan_period'].unique())

print(f"Data master siap:")
info = pd.DataFrame({
    'Aspek'  : ['Jumlah baris','Status tersisa','Bulan tersedia','Rentang data'],
    'Nilai'  : [f"{len(df_master):,}",
                'Pesanan Selesai',
                f"{len(bulan_semua)} bulan",
                f"{bulan_semua[0]} s.d. {bulan_semua[-1]}"]
})
display(info)


Memuat dan memproses dataset...
Data master siap:


,Aspek,Nilai
0,Jumlah baris,"31,880"
1,Status tersisa,Pesanan Selesai
2,Bulan tersedia,59 bulan
3,Rentang data,2020-05 s.d. 2025-03


---
## Sel 2 — Eksperimen Split 80:20 (Baseline)

Ini adalah konfigurasi baseline yang digunakan pada laporan utama. Training menggunakan 80% bulan pertama (47 bulan), testing menggunakan 20% bulan terakhir (12 bulan). Model memiliki data historis paling banyak untuk belajar dan periode testing mencakup satu siklus tahunan penuh.

In [3]:
hasil_80_20 = jalankan_eksperimen(0.80, df_master, label="80:20")


  EKSPERIMEN SPLIT 80:20
  Training : 47 bulan (2020-05 s.d. 2024-03)
  Testing  : 12 bulan (2024-04 s.d. 2025-03)
  Produk layak: 166 (ARIMA: 153)
  Melatih ARIMA (153 produk)...


c:\Skripsi FIX (5) - Copy\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (5) - Copy\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (5) - Copy\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (5) - Copy\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (5) - Copy\backend\.venv\

  ARIMA selesai: 96.7 detik, 7.2 MB
  Melatih LSTM (5 klaster, 3,201 sampel bersih)...
  LSTM selesai : 149.9 detik, 47.7 MB
  Evaluasi rolling 12 bulan testing...
  ARIMA   : MAE=2.021jt  RMSE=2.502jt  MAPE=28.6%
  LSTM    : MAE=2.025jt  RMSE=2.432jt  MAPE=24.2%
  Ensemble: MAE=1.958jt  RMSE=2.300jt  MAPE=25.2%


---
## Sel 3 — Eksperimen Split 70:30

Training menggunakan 70% bulan pertama, testing menggunakan 30% bulan terakhir. Periode testing lebih panjang dibanding baseline, sehingga evaluasi mencakup lebih banyak variasi kondisi penjualan namun data training berkurang.

In [4]:
hasil_70_30 = jalankan_eksperimen(0.70, df_master, label="70:30")


  EKSPERIMEN SPLIT 70:30
  Training : 41 bulan (2020-05 s.d. 2023-09)
  Testing  : 18 bulan (2023-10 s.d. 2025-03)
  Produk layak: 163 (ARIMA: 132)
  Melatih ARIMA (132 produk)...


c:\Skripsi FIX (5) - Copy\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (5) - Copy\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (5) - Copy\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (5) - Copy\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (5) - Copy\backend\.venv\

  ARIMA selesai: 92.3 detik, 6.8 MB
  Melatih LSTM (5 klaster, 2,486 sampel bersih)...
  LSTM selesai : 183.9 detik, 45.5 MB
  Evaluasi rolling 18 bulan testing...
  ARIMA   : MAE=2.383jt  RMSE=3.238jt  MAPE=25.0%
  LSTM    : MAE=3.472jt  RMSE=4.334jt  MAPE=33.0%
  Ensemble: MAE=2.556jt  RMSE=2.952jt  MAPE=26.6%


---
## Sel 4 — Eksperimen Split 60:40

Training menggunakan 60% bulan pertama, testing menggunakan 40% bulan terakhir. Pada titik ini, berkurangnya data training mulai mempengaruhi jumlah produk yang memenuhi syarat kelayakan minimum 15 bulan.

In [5]:
hasil_60_40 = jalankan_eksperimen(0.60, df_master, label="60:40")


  EKSPERIMEN SPLIT 60:40
  Training : 35 bulan (2020-05 s.d. 2023-03)
  Testing  : 24 bulan (2023-04 s.d. 2025-03)
  Produk layak: 151 (ARIMA: 116)
  Melatih ARIMA (116 produk)...


c:\Skripsi FIX (5) - Copy\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (5) - Copy\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (5) - Copy\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (5) - Copy\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (5) - Copy\backend\.venv\

  ARIMA selesai: 233.3 detik, 6.7 MB
  Melatih LSTM (5 klaster, 1,844 sampel bersih)...
  LSTM selesai : 273.4 detik, 45.0 MB
  Evaluasi rolling 24 bulan testing...
  ARIMA   : MAE=3.191jt  RMSE=4.090jt  MAPE=33.3%
  LSTM    : MAE=4.232jt  RMSE=4.990jt  MAPE=41.8%
  Ensemble: MAE=3.389jt  RMSE=4.151jt  MAPE=35.6%


---
## Sel 5 — Eksperimen Split 50:50

Training dan testing masing-masing menggunakan separuh data. Ini adalah titik keseimbangan di mana data training dan testing memiliki panjang yang sama, namun jumlah produk layak diperkirakan menurun signifikan dibanding konfigurasi dengan training lebih panjang.

In [6]:
hasil_50_50 = jalankan_eksperimen(0.50, df_master, label="50:50")


  EKSPERIMEN SPLIT 50:50
  Training : 29 bulan (2020-05 s.d. 2022-09)
  Testing  : 30 bulan (2022-10 s.d. 2025-03)
  Produk layak: 119 (ARIMA: 71)
  Melatih ARIMA (71 produk)...


c:\Skripsi FIX (5) - Copy\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (5) - Copy\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (5) - Copy\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (5) - Copy\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (5) - Copy\backend\.venv\

  ARIMA selesai: 143.8 detik, 6.6 MB
  Melatih LSTM (5 klaster, 1,086 sampel bersih)...
  LSTM selesai : 387.0 detik, 44.5 MB
  Evaluasi rolling 30 bulan testing...
  ARIMA   : MAE=2.822jt  RMSE=3.769jt  MAPE=37.2%
  LSTM    : MAE=3.804jt  RMSE=4.886jt  MAPE=38.2%
  Ensemble: MAE=2.708jt  RMSE=3.492jt  MAPE=33.7%


---
## Sel 6 — Eksperimen Split 40:60

Training menggunakan 40% bulan pertama, testing menggunakan 60% bulan terakhir. Data training yang semakin sedikit menyebabkan banyak produk tidak memenuhi syarat minimum riwayat, sehingga jumlah produk yang dapat dimodelkan diperkirakan turun tajam.

In [7]:
hasil_40_60 = jalankan_eksperimen(0.40, df_master, label="40:60")


  EKSPERIMEN SPLIT 40:60
  Training : 23 bulan (2020-05 s.d. 2022-03)
  Testing  : 36 bulan (2022-04 s.d. 2025-03)
  Produk layak: 72 (ARIMA: 23)
  Melatih ARIMA (23 produk)...


c:\Skripsi FIX (5) - Copy\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Skripsi FIX (5) - Copy\backend\.venv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  ARIMA selesai: 45.7 detik, 6.4 MB
  Melatih LSTM (5 klaster, 454 sampel bersih)...
  LSTM selesai : 146.4 detik, 17.3 MB
  Evaluasi rolling 36 bulan testing...
  ARIMA   : MAE=2.237jt  RMSE=2.816jt  MAPE=38.3%
  LSTM    : MAE=3.055jt  RMSE=4.144jt  MAPE=41.4%
  Ensemble: MAE=2.012jt  RMSE=2.940jt  MAPE=34.0%


---
## Sel 7 — Eksperimen Split 30:70

Training menggunakan 30% bulan pertama (sekitar 17-18 bulan), testing menggunakan 70% bulan terakhir. Dengan data training yang sangat terbatas, banyak produk tidak memenuhi syarat minimum 15 bulan untuk LSTM maupun 20 bulan untuk ARIMA.

In [8]:
hasil_30_70 = jalankan_eksperimen(0.30, df_master, label="30:70")


  EKSPERIMEN SPLIT 30:70
  [SKIP] Data training terlalu sedikit: 17 bulan


---
## Sel 8 — Eksperimen Split 20:80

Training menggunakan hanya 20% bulan pertama (sekitar 11-12 bulan), testing menggunakan 80% bulan terakhir. Ini adalah konfigurasi paling ekstrem dengan data training paling sedikit. Kemungkinan besar jumlah produk layak akan sangat terbatas karena syarat minimum 15 bulan hampir tidak terpenuhi.

In [9]:
hasil_20_80 = jalankan_eksperimen(0.20, df_master, label="20:80")


  EKSPERIMEN SPLIT 20:80
  [SKIP] Data training terlalu sedikit: 11 bulan


---
## Sel 9 — Tabel Perbandingan Hasil Seluruh Eksperimen

Sel ini merangkum seluruh hasil eksperimen ke dalam beberapa tabel perbandingan yang dapat digunakan langsung sebagai referensi penulisan Bab 4 laporan skripsi.


In [16]:
# =====================================================================
# KUMPULKAN SEMUA HASIL
# =====================================================================
semua = [hasil_80_20, hasil_70_30, hasil_60_40, hasil_50_50,
         hasil_40_60, hasil_30_70, hasil_20_80]
semua = [h for h in semua if h is not None]

if not semua:
    print("Tidak ada hasil eksperimen yang tersedia.")
else:
    # =====================================================================
    # TABEL 1
    # =====================================================================
    print("=" * 70)
    print("TABEL 1 — Informasi Partisi Data per Rasio Split")
    print("=" * 70)
    t1 = pd.DataFrame([{
        'Rasio Split'      : h['label'],
        'Bulan Training'   : h['n_bulan_train'],
        'Periode Training' : h['periode_train'],
        'Bulan Testing'    : h['n_bulan_test'],
        'Periode Testing'  : h['periode_test'],
        'Produk Layak'     : h['n_produk_layak'],
        'Produk ARIMA'     : h['n_produk_arima'],
        'Sampel LSTM'      : h['n_sampel_lstm'],
    } for h in semua])
    display(t1)

    # =====================================================================
    # TABEL 2
    # =====================================================================
    print()
    print("=" * 70)
    print("TABEL 2 — Metrik Evaluasi Model ARIMA per Rasio Split")
    print("=" * 70)
    best_mae_a = min(h['mae_arima'] for h in semua)
    t2 = pd.DataFrame([{
        'Rasio Split' : h['label'],
        'MAE (Rp)'    : f"{h['mae_arima']:,.0f}",
        'RMSE (Rp)'   : f"{h['rmse_arima']:,.0f}",
        'MAPE (%)'    : f"{h['mape_arima']:.1f}",
        'Waktu (det)' : h['t_arima'],
        'Memori (MB)' : h['mem_arima'],
        'MAE Terbaik?': 'TERBAIK' if abs(h['mae_arima'] - best_mae_a) < 1 else '',
    } for h in semua])
    display(t2)

    # =====================================================================
    # TABEL 3
    # =====================================================================
    print()
    print("=" * 70)
    print("TABEL 3 — Metrik Evaluasi Model LSTM per Rasio Split")
    print("=" * 70)
    best_mae_l = min(h['mae_lstm'] for h in semua)
    t3 = pd.DataFrame([{
        'Rasio Split' : h['label'],
        'MAE (Rp)'    : f"{h['mae_lstm']:,.0f}",
        'RMSE (Rp)'   : f"{h['rmse_lstm']:,.0f}",
        'MAPE (%)'    : f"{h['mape_lstm']:.1f}",
        'Waktu (det)' : h['t_lstm'],
        'Memori (MB)' : h['mem_lstm'],
        'MAE Terbaik?': 'TERBAIK' if abs(h['mae_lstm'] - best_mae_l) < 1 else '',
    } for h in semua])
    display(t3)

    # =====================================================================
    # TABEL 4
    # =====================================================================
    print()
    print("=" * 70)
    print("TABEL 4 — Metrik Evaluasi Ensemble Adaptif per Rasio Split")
    print("=" * 70)
    best_mae_e = min(h['mae_ens'] for h in semua)
    t4 = pd.DataFrame([{
        'Rasio Split' : h['label'],
        'MAE (Rp)'    : f"{h['mae_ens']:,.0f}",
        'RMSE (Rp)'   : f"{h['rmse_ens']:,.0f}",
        'MAPE (%)'    : f"{h['mape_ens']:.1f}",
        'MAE Terbaik?': 'TERBAIK' if abs(h['mae_ens'] - best_mae_e) < 1 else '',
    } for h in semua])
    display(t4)

    # =====================================================================
    # TABEL 5
    # =====================================================================
    print()
    print("=" * 70)
    print("TABEL 5 — Perbandingan Lengkap: Semua Model x Semua Rasio")
    print("=" * 70)
    rows = []
    for h in semua:
        rows.append({
            'Rasio'           : h['label'],
            'ARIMA MAE (Rp)'  : f"{h['mae_arima']:,.0f}",
            'ARIMA RMSE (Rp)' : f"{h['rmse_arima']:,.0f}",
            'ARIMA MAPE (%)'  : f"{h['mape_arima']:.1f}",
            'LSTM MAE (Rp)'   : f"{h['mae_lstm']:,.0f}",
            'LSTM RMSE (Rp)'  : f"{h['rmse_lstm']:,.0f}",
            'LSTM MAPE (%)'   : f"{h['mape_lstm']:.1f}",
            'ENS MAE (Rp)'    : f"{h['mae_ens']:,.0f}",
            'ENS RMSE (Rp)'   : f"{h['rmse_ens']:,.0f}",
            'ENS MAPE (%)'    : f"{h['mape_ens']:.1f}",
        })
    display(pd.DataFrame(rows))

    # =====================================================================
    # TABEL 6
    # =====================================================================
    print()
    print("=" * 70)
    print("TABEL 6 — Model Terbaik (MAE Terendah) per Rasio Split")
    print("=" * 70)
    rows6 = []
    for h in semua:
        maes  = {'ARIMA': h['mae_arima'], 'LSTM': h['mae_lstm'], 'Ensemble': h['mae_ens']}
        mapes = {'ARIMA': h['mape_arima'], 'LSTM': h['mape_lstm'], 'Ensemble': h['mape_ens']}
        best_model = min(maes, key=maes.get)
        rows6.append({
            'Rasio Split'      : h['label'],
            'Model Terbaik'    : best_model,
            'MAE Terbaik (Rp)' : f"{maes[best_model]:,.0f}",
            'MAPE Terbaik (%)' : f"{mapes[best_model]:.1f}",
            'Produk Layak'     : h['n_produk_layak'],
            'Bulan Training'   : h['n_bulan_train'],
            'Bulan Testing'    : h['n_bulan_test'],
        })
    display(pd.DataFrame(rows6))

    # =====================================================================
    # TABEL 7
    # =====================================================================
    print()
    print("=" * 70)
    print("TABEL 7 — Detail Prediksi Ensemble vs Aktual per Bulan")
    print("          (untuk setiap rasio split)")
    print("=" * 70)
    for h in semua:
        print(f"\nRasio {h['label']} — {h['n_bulan_test']} bulan testing:")
        dl = h['df_log'].copy()
        dl['Aktual (jt)']   = (dl['aktual'] / 1e6).round(1)
        dl['ARIMA (jt)']    = (dl['pred_arima'] / 1e6).round(1)
        dl['LSTM (jt)']     = (dl['pred_lstm'] / 1e6).round(1)
        dl['Ensemble (jt)'] = (dl['pred_ens'] / 1e6).round(1)
        dl['Error Ens %']   = (
            (dl['pred_ens'] - dl['aktual']).abs()
            / dl['aktual'].clip(lower=1) * 100
        ).round(0).astype(int)
        display(dl[['bulan', 'Aktual (jt)', 'ARIMA (jt)',
                     'LSTM (jt)', 'Ensemble (jt)', 'Error Ens %']])

TABEL 1 — Informasi Partisi Data per Rasio Split


,Rasio Split,Bulan Training,Periode Training,Bulan Testing,Periode Testing,Produk Layak,Produk ARIMA,Sampel LSTM
0,80:20,47,2020-05 s.d. 2024-03,12,2024-04 s.d. 2025-03,166,153,3201
1,70:30,41,2020-05 s.d. 2023-09,18,2023-10 s.d. 2025-03,163,132,2486
2,60:40,35,2020-05 s.d. 2023-03,24,2023-04 s.d. 2025-03,151,116,1844
3,50:50,29,2020-05 s.d. 2022-09,30,2022-10 s.d. 2025-03,119,71,1086
4,40:60,23,2020-05 s.d. 2022-03,36,2022-04 s.d. 2025-03,72,23,454



TABEL 2 — Metrik Evaluasi Model ARIMA per Rasio Split


,Rasio Split,MAE (Rp),RMSE (Rp),MAPE (%),Waktu (det),Memori (MB),MAE Terbaik?
0,80:20,"2,021,431","2,501,649",28.6,96.70,7.20,TERBAIK
1,70:30,"2,383,436","3,237,769",25.0,92.30,6.80,
2,60:40,"3,190,713","4,090,336",33.3,233.30,6.70,
3,50:50,"2,822,083","3,769,084",37.2,143.80,6.60,
4,40:60,"2,237,345","2,816,380",38.3,45.70,6.40,



TABEL 3 — Metrik Evaluasi Model LSTM per Rasio Split


...

,bulan,Aktual (jt),ARIMA (jt),LSTM (jt),Ensemble (jt),Error Ens %
0,2023-04,20.70,10.80,9.80,10.20,51
1,2023-05,15.30,21.60,15.80,18.10,18
2,2023-06,17.20,13.20,9.30,10.80,37
3,2023-07,14.90,20.30,13.90,17.30,16
4,2023-08,19.30,19.60,14.10,17.00,12
5,2023-09,21.20,20.40,14.30,17.60,17
6,2023-10,19.30,21.40,15.50,18.80,3
7,2023-11,19.60,21.60,16.60,19.40,1
8,2023-12,13.00,22.00,19.10,20.60,59
9,2024-01,16.30,20.50,21.10,20.80,28



Rasio 50:50 — 30 bulan testing:


,bulan,Aktual (jt),ARIMA (jt),LSTM (jt),Ensemble (jt),Error Ens %
0,2022-10,22.80,21.00,15.10,17.50,23
1,2022-11,18.90,21.90,14.90,17.70,6
2,2022-12,17.90,21.70,13.10,16.60,7
3,2023-01,16.70,21.30,14.20,17.80,7
4,2023-02,13.10,20.50,13.20,16.70,28
5,2023-03,18.70,18.70,12.80,15.60,17
6,2023-04,18.50,9.10,6.30,7.70,58
7,2023-05,13.10,18.30,9.70,14.10,7
8,2023-06,14.20,11.20,6.20,8.90,38
9,2023-07,11.40,16.90,9.10,13.40,18



Rasio 40:60 — 36 bulan testing:


,bulan,Aktual (jt),ARIMA (jt),LSTM (jt),Ensemble (jt),Error Ens %
0,2022-04,15.70,19.30,9.40,13.40,15
1,2022-05,17.90,9.90,4.50,6.70,63
2,2022-06,9.20,11.50,4.90,7.60,18
3,2022-07,9.60,10.80,4.60,8.20,14
4,2022-08,11.00,13.90,7.30,11.10,1
5,2022-09,17.60,13.10,7.70,10.70,39
6,2022-10,14.60,15.00,8.40,11.90,18
7,2022-11,13.00,15.30,9.10,12.30,6
8,2022-12,12.20,15.00,8.60,11.80,4
9,2023-01,10.70,14.70,8.40,11.40,6


---
## Sel 10 — Analisis Kuantitatif dan Rekomendasi Rasio Split

Sel ini menghasilkan analisis otomatis berdasarkan hasil numerik eksperimen untuk membantu penentuan rasio split yang paling optimal.


In [ ]:
if semua:
    print("=" * 70)
    print("ANALISIS OTOMATIS BERDASARKAN HASIL EKSPERIMEN")
    print("=" * 70)

    # Cari rasio dengan MAE ensemble terbaik
    best_ens = min(semua, key=lambda h: h['mae_ens'])
    best_arima = min(semua, key=lambda h: h['mae_arima'])
    best_lstm  = min(semua, key=lambda h: h['mae_lstm'])

    analisis = pd.DataFrame([{
        'Kategori'      : 'MAE Ensemble Terbaik',
        'Rasio Split'   : best_ens['label'],
        'Nilai'         : f"Rp{best_ens['mae_ens']:,.0f}",
        'Bulan Training': best_ens['n_bulan_train'],
        'Produk Layak'  : best_ens['n_produk_layak'],
    }, {
        'Kategori'      : 'MAE ARIMA Terbaik',
        'Rasio Split'   : best_arima['label'],
        'Nilai'         : f"Rp{best_arima['mae_arima']:,.0f}",
        'Bulan Training': best_arima['n_bulan_train'],
        'Produk Layak'  : best_arima['n_produk_layak'],
    }, {
        'Kategori'      : 'MAE LSTM Terbaik',
        'Rasio Split'   : best_lstm['label'],
        'Nilai'         : f"Rp{best_lstm['mae_lstm']:,.0f}",
        'Bulan Training': best_lstm['n_bulan_train'],
        'Produk Layak'  : best_lstm['n_produk_layak'],
    }])
    display(analisis)

    # Tren MAE vs rasio split
    print()
    print("Tren MAE Ensemble seiring bertambahnya data training:")
    tren = pd.DataFrame([{
        'Rasio Split'    : h['label'],
        'Bulan Training' : h['n_bulan_train'],
        'Bulan Testing'  : h['n_bulan_test'],
        'Produk Layak'   : h['n_produk_layak'],
        'MAE Ens (Rp)'   : f"{h['mae_ens']:,.0f}",
        'MAPE Ens (%)'   : f"{h['mape_ens']:.1f}",
        'Interpretasi'   : (
            'Baseline (laporan)' if h['label']=='80:20' else
            'Training sangat sedikit' if h['n_bulan_train'] < 20 else
            'Training sedikit' if h['n_bulan_train'] < 30 else
            'Testing terlalu panjang' if h['n_bulan_test'] > 30 else
            'Seimbang'
        )
    } for h in semua])
    display(tren)

    print()
    print("=" * 70)
    print("KESIMPULAN REKOMENDASI")
    print("=" * 70)
    print(f"Berdasarkan eksperimen terhadap {len(semua)} rasio split yang diuji:")
    print()
    print(f"  1. Rasio dengan MAE Ensemble terbaik : {best_ens['label']}")
    print(f"     MAE = Rp{best_ens['mae_ens']:,.0f}  |  MAPE = {best_ens['mape_ens']:.1f}%")
    print(f"     Bulan training = {best_ens['n_bulan_train']}  |  Produk layak = {best_ens['n_produk_layak']}")
    print()
    h_baseline = next((h for h in semua if h['label']=='80:20'), None)
    if h_baseline:
        selisih = best_ens['mae_ens'] - h_baseline['mae_ens']
        arah = 'lebih baik' if selisih < 0 else 'lebih buruk'
        print(f"  2. Dibandingkan baseline 80:20:")
        print(f"     MAE baseline = Rp{h_baseline['mae_ens']:,.0f}")
        print(f"     Selisih      = Rp{abs(selisih):,.0f} ({arah})")
    print()
    print("  3. Pertimbangan tambahan selain akurasi:")
    print("     - Semakin banyak bulan training -> semakin banyak produk layak")
    print("     - Semakin banyak bulan testing  -> evaluasi lebih representatif")
    print("     - Rasio 80:20 memastikan 1 siklus kalender penuh sebagai testing")
    print("     - Rasio dengan training < 20 bulan: terlalu sedikit produk layak")


ANALISIS OTOMATIS BERDASARKAN HASIL EKSPERIMEN


,Kategori,Rasio Split,Nilai,Bulan Training,Produk Layak
0,MAE Ensemble Terbaik,80:20,"Rp1,958,412",47,166
1,MAE ARIMA Terbaik,80:20,"Rp2,021,431",47,166
2,MAE LSTM Terbaik,80:20,"Rp2,025,136",47,166



Tren MAE Ensemble seiring bertambahnya data training:


,Rasio Split,Bulan Training,Bulan Testing,Produk Layak,MAE Ens (Rp),MAPE Ens (%),Interpretasi
0,80:20,47,12,166,"1,958,412",25.2,Baseline (laporan)
1,70:30,41,18,163,"2,555,640",26.6,Seimbang
2,60:40,35,24,151,"3,388,963",35.6,Seimbang
3,50:50,29,30,119,"2,708,012",33.7,Training sedikit
4,40:60,23,36,72,"2,011,517",34.0,Training sedikit



KESIMPULAN REKOMENDASI
Berdasarkan eksperimen terhadap 5 rasio split yang diuji:

  1. Rasio dengan MAE Ensemble terbaik : 80:20
     MAE = Rp1,958,412  |  MAPE = 25.2%
     Bulan training = 47  |  Produk layak = 166

  2. Dibandingkan baseline 80:20:
     MAE baseline = Rp1,958,412
     Selisih      = Rp0 (lebih buruk)

  3. Pertimbangan tambahan selain akurasi:
     - Semakin banyak bulan training -> semakin banyak produk layak
     - Semakin banyak bulan testing  -> evaluasi lebih representatif
     - Rasio 80:20 memastikan 1 siklus kalender penuh sebagai testing
     - Rasio dengan training < 20 bulan: terlalu sedikit produk layak


: 